# Thrifted Jeans follow-up audit

This notebook is a separated follow-up to `JEANS_STRATEGY_AUDIT.md`. It replays the omitted hybrid-without-EMA ablation, corrects the weak-state best-trade test, uses additive absolute-change path bootstraps, and compares paired candidate performance. It is research-only: production strategy and simulator files are not edited.

The simulator convention charges the return from day `t` to `t+1` to the position held on day `t`. Accordingly, the requested flat-day-zero Candidate A is AUD 37,760, while the familiar AUD 38,136 is the full `+800` reference held from day zero. Both references are retained explicitly.

In [1]:
from pathlib import Path
import sys
import pandas as pd

AUDIT_DIR = Path.cwd()
if AUDIT_DIR.name != 'thrifted_jeans':
    AUDIT_DIR = Path(r'D:/Documents/Algojam/research/thrifted_jeans')
sys.path.insert(0, str(AUDIT_DIR))
from jeans_followup_audit import run_followup

OUTPUT_DIR = AUDIT_DIR / 'followup_outputs'
FIGURE_DIR = AUDIT_DIR / 'followup_figures'

In [2]:
# Reproducible audit run. Repetitions are explicit in the manifest.
# Reuse a completed run when present; deleting followup_outputs forces a rebuild.
import json
manifest_path = OUTPUT_DIR / 'followup_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected = (1000, 300, 300, 1000)
    observed = (manifest.get('additive_repetitions_per_block'), manifest.get('drift_repetitions_per_assumption'), manifest.get('percentage_secondary_repetitions_per_block'), manifest.get('familywise_repetitions'))
else:
    manifest, observed, expected = {}, (), (1000, 300, 300, 1000)
if observed == expected:
    RESULT = {
        'comparison': pd.read_csv(OUTPUT_DIR / 'exact_candidate_comparison.csv').set_index('Candidate'),
        'correctness': pd.read_csv(OUTPUT_DIR / 'correctness_checks.csv'),
        'manifest': manifest,
    }
    print('Loaded the completed reproducible audit run from followup_outputs/.')
else:
    RESULT = run_followup(
        output_dir=OUTPUT_DIR,
        figure_dir=FIGURE_DIR,
        additive_repetitions=1000,
        drift_repetitions=300,
        percentage_repetitions=300,
        familywise_repetitions=1000,
    )
RESULT['comparison'][['P&L', 'Incremental vs always long', 'Incremental vs K2', 'Strong P&L', 'Weak P&L', 'Max drawdown', 'Turnover']].round(2)

Loaded the completed reproducible audit run from followup_outputs/.


,P&L,Incremental vs always long,Incremental vs K2,Strong P&L,Weak P&L,Max drawdown,Turnover
Candidate,,,,,,,
always_long,37760.0,-376.0,-62256.0,13800.0,23960.0,-44856.0,800
simple_K2,100016.0,61880.0,0.0,76056.0,23960.0,-26672.0,29600
hybrid_Kalman_no_EMA,119584.0,81448.0,19568.0,95624.0,23960.0,-35512.0,32800
corrected_hybrid,151584.0,113448.0,51568.0,95624.0,55960.0,-14536.0,53600
original_hybrid,162496.0,124360.0,62480.0,95624.0,66872.0,-14536.0,58400
strong_direction_flat_weak,95624.0,57488.0,-4392.0,95624.0,0.0,-29864.0,38400
strong_direction_long_weak,119584.0,81448.0,19568.0,95624.0,23960.0,-35512.0,32800
strong_direction_hold_weak,80200.0,42064.0,-19816.0,95624.0,-15424.0,-40680.0,26400
strong_direction_latest_reversal_weak,134592.0,96456.0,34576.0,95624.0,38968.0,-19960.0,100000


In [3]:
# Core acceptance checks and a compact hand-off preview.
comparison = RESULT['comparison']
assert comparison.loc['simple_K2', 'P&L'] == 100016.0
assert comparison.loc['hybrid_Kalman_no_EMA', 'P&L'] == 119584.0
assert comparison.loc['corrected_hybrid', 'P&L'] == 151584.0
assert comparison.loc['original_hybrid', 'P&L'] == 162496.0
assert comparison.loc['always_long', 'P&L'] == 37760.0
assert RESULT['manifest']['reference_pnls']['always_long_full_reference'] == 38136.0
assert RESULT['correctness']['Value'].astype(str).str.lower().eq('true').all()
print('Notebook checks passed.')
print('Generated CSVs:', len(list(OUTPUT_DIR.glob('*.csv'))))
print('Generated figures:', len(list(FIGURE_DIR.glob('*.png'))))

Notebook checks passed.
Generated CSVs: 18
Generated figures: 6
